# 01 Exploratory Data Analysis: UCI Heart Disease

This notebook creates the first real data-analysis layer for the clinical risk modeling portfolio project. It documents and explores the UCI Heart Disease dataset without training machine learning models.

## Project Context

Project line: Biomedical Data Science for Personalized Medicine.

Current phase: data documentation and exploratory data analysis only. This notebook is educational and does not present a diagnostic medical tool or clinical decision support system.

## Dataset Citation

Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X

Dataset page: https://archive.ics.uci.edu/dataset/45/heart+disease

License: CC BY 4.0, according to the UCI dataset page.

## Ethical Note

The dataset is public, but it originates from clinical data. Interpretations should remain cautious, descriptive, and limited to this dataset. Do not use this notebook for medical diagnosis, treatment decisions, or causal claims.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import (  # noqa: E402
    HEART_DISEASE_COLUMNS,
    binarize_heart_disease_target,
    clean_missing_values,
    load_heart_disease_data,
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

## Load Data

Expected raw file path: `data/raw/processed.cleveland.data`. The file is intentionally not included in this repository.

In [ ]:
def load_dataset_or_explain(path: Path) -> pd.DataFrame | None:
    """Load the local UCI file or print manual download instructions."""
    if not path.exists():
        print("Raw data file not found.")
        print("Download the UCI Heart Disease dataset from:")
        print("https://archive.ics.uci.edu/dataset/45/heart+disease")
        print("Place the processed Cleveland file at:")
        print(path.relative_to(PROJECT_ROOT))
        return None

    return clean_missing_values(load_heart_disease_data(path))


df = load_dataset_or_explain(RAW_DATA_PATH)
HEART_DISEASE_COLUMNS

## Basic DataFrame Inspection

In [ ]:
if df is not None:
    print(df.head())
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

In [ ]:
if df is not None:
    print(df.dtypes.to_frame("dtype"))

## Missing Values and Duplicates

In [ ]:
if df is not None:
    missing = (
        df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_percent=lambda table: table["missing_count"] / len(df) * 100)
    )
    print(missing.sort_values("missing_count", ascending=False))
    print(f"Duplicate rows: {df.duplicated().sum()}")

## Target Distribution

The original target coding uses `0`, `1`, `2`, `3`, and `4`. For future binary educational modeling, values greater than `0` may be mapped to `1`. This is a dataset-label transformation, not a medical diagnosis.

In [ ]:
if df is not None:
    target_counts = df["target"].value_counts(dropna=False).sort_index()
    binary_target = binarize_heart_disease_target(df)["target"]
    binary_counts = binary_target.value_counts(dropna=False).sort_index()

    print(target_counts.rename("original_target_count").to_frame())
    print(binary_counts.rename("binary_target_count").to_frame())

## Numeric and Categorical Summaries

In [ ]:
numeric_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_columns = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
    "target",
]

if df is not None:
    print(df[numeric_columns].describe().T)
    categorical_summary = {
        column: df[column].value_counts(dropna=False).sort_index()
        for column in categorical_columns
    }
    for column, counts in categorical_summary.items():
        print(f"\n{column}")
        print(counts.rename("count").to_frame())

## Basic Plots

In [ ]:
if df is not None:
    ax = df["target"].value_counts().sort_index().plot(kind="bar")
    ax.set_title("Original Target Distribution")
    ax.set_xlabel("Original target code")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "target_distribution.png", dpi=150)
    plt.show()

    axes = df[numeric_columns].hist(figsize=(10, 7), bins=20)
    for axis in axes.flatten():
        axis.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "numeric_variable_histograms.png", dpi=150)
    plt.show()

## Cautious Biomedical Interpretation

EDA should focus on data quality, distributions, missingness, and whether variables are plausible for later baseline modeling. Descriptive differences across target groups should be treated as observations within this dataset only. They should not be interpreted as causal effects or as evidence that the notebook can diagnose disease.

## Limitations

- The dataset is historical and may not reflect current clinical practice.
- The processed Cleveland file is a reduced 14-variable version of the original dataset.
- Several predictors are encoded categories that require careful documentation.
- Missing values must be handled transparently before modeling.
- This analysis is educational and not clinically deployable.

## Next Steps

1. Confirm the raw file source, download date, and local placement.
2. Review missingness and coded categorical variables.
3. Decide on preprocessing choices before modeling.
4. Add simple baseline modeling only in a later phase.